In [1]:
from dotvectordb.plugins.llm_providers.openai_llms import OpenAiAgent
from dotvectordb.plugins.llm_providers.gemini_llms import GeminiAgent
from dotvectordb.plugins.llm_providers.openrouter_llms import OpenRouterAgent

# All agents are asynchronous and follow the same interface. You can use them interchangeably in your code.
GPT = OpenAiAgent()
response, tokens = await GPT.generate(
    prompt="Hello, how are you?",)
print("OpenAI's Response:", response)
print("Generated a total of", tokens, "tokens.")
print("--------------------------------------------------")

# Gemini = GeminiAgent()
# response, tokens = await Gemini.generate(
#     prompt="Hello, how are you?",)
# print("Gemini's Response:", response)
# print("Generated a total of", tokens, "tokens.")
# print("--------------------------------------------------")

# OpenRouter = OpenRouterAgent()
# response, tokens = await OpenRouter.generate(
#     prompt="Hello, how are you?",)
# print("OpenRouter's Response:", response)
# print("Generated a total of", tokens, "tokens.")
print("--------------------------------------------------")

OpenAI's Response: {'answer': "I'm just a computer program, so I don't have feelings, but I'm here and ready to help you! How can I assist you today?"}
Generated a total of 32 tokens.
--------------------------------------------------
--------------------------------------------------


In [ ]:
from dotvectordb.plugins.embedding_providers.openai_embeddings import OpenAIEmbeddingModel

# Similarly, all embedding providers are asynchronous and follow the same interface. You can use them interchangeably in your code as well.
GPTEmbedding = OpenAIEmbeddingModel()
embeddings, tokens = await GPTEmbedding.embed(
    text="Hello, how are you?",)
print("OpenAI's Embeddings:", embeddings)
print("Ingested a total of", tokens, "tokens.")
print("--------------------------------------------------")


OpenAI's Embeddings: [-0.0309906  -0.00079536 -0.03152466 ...  0.02272034 -0.01947021
  0.00437164]
Ingested a total of 6 tokens.
--------------------------------------------------


In [ ]:
from dotvectordb.plugins.chunk_functions.FixedSizeChunker import FixedSizeChunker
from dotvectordb.plugins.hash_functions.hashfuncs import SHA256HashingFunc

SampleText = """This is a sample text that we will use to demonstrate the chunking and hashing functionality of the FixedSizeChunker class. The chunker will split this text into smaller chunks of a specified size, and then we will hash each chunk using the SHA256 hashing function."""

hasher = SHA256HashingFunc()
chunker = FixedSizeChunker(hash_func=hasher)
chunks = chunker.chunk_text(SampleText)
print("Chunks and their corresponding hashes:")
print("--------------------------------------------------")
for chunk in chunks:
    print(chunk)
    print("--------------------------------------------------")

Chunks and their corresponding hashes:
--------------------------------------------------
{'chunk_index': 0, 'tokens': 54, 'content': 'This is a sample text that we will use to demonstrate the chunking and hashing functionality of the FixedSizeChunker class. The chunker will split this text into smaller chunks of a specified size, and then we will hash each chunk using the SHA256 hashing function.', 'chunk_hash': 4458880990588974601}
--------------------------------------------------


In [ ]:
from dotvectordb.DotVDB import DotVectorDB

# Initialize the DotVectorDB with the specified path, dimension, embedding model, and language model.
db = DotVectorDB(path="test_db5.db", DIM=1536, embedding_model=GPTEmbedding, language_model=GPT)

# Open the database, which will create it if it doesn't exist.
await db.open()
await db.display_tables()

Table: embedding_model
  Column: id (Type: TEXT)
  Column: name (Type: TEXT)
  Column: description (Type: JSON)
Table: source
  Column: id (Type: TEXT)
  Column: name (Type: TEXT)
  Column: description (Type: JSON)
  Column: created_at (Type: TIMESTAMP)
  Column: num_chunks (Type: INT)
  Column: num_entities (Type: INT)
  Column: num_relations (Type: INT)
  Column: raw_corpus_tokens (Type: INT)
  Column: entity_construct_tokens (Type: INT)
  Column: entity_deduplication_tokens (Type: INT)
  Column: relation_construct_tokens (Type: INT)
Table: chunk
  Column: chunk_id (Type: INTEGER)
  Column: source_id (Type: TEXT)
  Column: content (Type: TEXT)
  Column: embedding_model_id (Type: TEXT)
  Column: is_embedded (Type: BOOLEAN)
  Column: are_entities_extracted (Type: BOOLEAN)
  Column: are_relations_extracted (Type: BOOLEAN)
Table: entity_types
  Column: entity_type (Type: TEXT)
  Column: type_description (Type: TEXT)
  Column: num_entities (Type: INT)
Table: entity
  Column: entity_id (Ty

In [ ]:
from dotvectordb.plugins.loaders.TextConverter import DocumentTextExtractor
from dotvectordb.plugins.chunk_functions.FixedSizeChunker import FixedSizeChunker
from dotvectordb.plugins.hash_functions.hashfuncs import SHA256HashingFunc

# Loads a sample text file from the tests folder called shrek.txt and converts it to a string
text_converter = DocumentTextExtractor(directories=["./tests"])
results = text_converter.controller()
print(results)

# Chunks the text into fixed-size pieces
chunker = FixedSizeChunker(hash_func=SHA256HashingFunc())
chunks = chunker.chunk_text(results['tests/shrek.txt'])
print("Number of chunks created:", len(chunks))

{'tests/shrek.txt': '{Man} Once upon a time there was a lovely princess.\nBut she had an enchantment upon her of a fearful sort which could only \nbe broken by love\'s first kiss.\nShe was locked away in a castle guarded by a terrible fire-breathing \ndragon.\nMany brave knigts had attempted to free her from this dreadful prison, \nbut non prevailed.\nShe waited in the dragon\'s keep in the highest room of the tallest \ntower for her true love and true love\'s first kiss.\n{Laughing} \nLike that\'s ever gonna happen.\n{Paper Rusting, Toilet Flushes}\nWhat a load of - \n\nSomebody once told me the world is gonna roll me\nI ain\'t the sharpest tool in the shed\nShe was lookin\' kind of dumb with her finger and her thumb\nIn the shape of an "L" on her forehead\nThe years start comin\' and they don\'t stop comin\'\nFed to the rules and hit the ground runnin\'\nDidn\'t make sense not to live for fun\nYour brain gets smart but your head gets dumb\nSo much to do so much to see\nSo what\'s wro

In [6]:
# Now, it's time ingest all the chunks into the database.
await db.ingest_documents(input_document_list=["./tests"], verbose=True)

data = await db.get_unprocessed_metadata("chunk")
print("Unprocessed metadata:", data)

Document: tests/shrek.txt
Total chunks created: 17
Starting to add embeddings for each chunk to the database...
All embeddings added successfully.
Unprocessed metadata: []


In [7]:
retrieved_content = await db.vector_search("Shrek", table="chunk", top_k=3)
print("Retrieved content:", retrieved_content)

Retrieved content: ([{'vector_rowid': 1445511819865618881, 'chunk_id': 1445511819865618881, 'source_id': '-8917888661152971970', 'content': 'princess.\n-We? Donkey, there\'s no "we". There\'s no "our". There\'s just me and my \nswamp. The first thing I\'m gonna do is build a ten-foot wall arond my \nland.\n-You cut me deep, Shrek. You cut me real deep just now. You know what \nI think? I think this whole wall thing is just a way to keep somebody \nout.\n-No, do ya think?\n-Are you hidin\' something?\n-Never mind, Donkey.\n-Oh, this is another one of those onion things, isn\'t it?\n-No, this is one of those drop-it and leave-it alone things.\n-Why don\'t you want to talk about it?\n-Why do you want to talk about it?\n-Why are you blocking?\n-I\'m not blocking.\n-Oh, yes, you are.\n-Donkey, I\'m warning you.\n-Who you trying to keep out?\n-Everyone! Okay?\n-Oh, now we\'re gettin\' somewhere.\n-Oh! For the love of Pete!\n-What\'s your problem? What you got against the whole world anyway?\

In [8]:
await db.extract_and_embed_entities(verbose=True)

Found 0 chunks waiting for entity extraction.


await db.

In [9]:
await db.extract_and_embed_relationships(verbose=True)

Found 0 chunks waiting for relationship extraction.
Found 39 unprocessed relationships. Starting to add embeddings...
Sending batch of 39 relationships to embedding model...


Embedding relationships:   0%|          | 0/39 [00:00<?, ?it/s]

Prompt for relationship 3295705859478370061: {'src_entity_id': '6419349434901819561', 'dst_entity_id': '-5265323203871822751', 'relation_description': "Shrek's quest to rescue Princess Fiona is motivated by his desire to reclaim his swamp, which was filled with fairy tale creatures by Farquaad.<SEP>Shrek interrupts Fiona's wedding to express his love for her, indicating that he believes she is not marrying for true love.<SEP>Shrek mentions that they need to get to DuLoc, which is where Lord Farquaad resides.", 'questions': '["Why does Shrek want to rescue Princess Fiona?", "What happened to Shrek\'s swamp?", "Why does Shrek object to Fiona\'s wedding?", "What does Shrek believe about Fiona\'s feelings towards her groom?", "Where do Shrek and his companions need to go?", "What is the significance of DuLoc in relation to Lord Farquaad?"]'}
Prompt for relationship -7121838788470248392: {'src_entity_id': '6419349434901819561', 'dst_entity_id': '3358813184437975775', 'relation_description':

Embedding relationships: 100%|██████████| 39/39 [00:12<00:00,  3.12it/s]
